## Importing các libraries cần thiết
Sử dụng các python libraries BeautifulSoup, json và requests để làm web scraping. Đồng thời import os để lưu dữ liệu về máy và datetime để stamp dữ liệu.

In [1]:
import requests

from bs4 import BeautifulSoup

import os
import json

import time
from datetime import datetime

## Thu thập báo
Bắt đầu từ 1 trang gốc, rồi recursively tìm các trang khác qua tag "a". Đồng thời kiểm tra xem là trang chính hay là trang tin mới thật bằng cách so sánh page_type (page_type = article => trang tin tức).


Sử dụng set chứa các url để đảm bảo không bị nhét trùng vào bộ scraper.

In [ ]:
pages = []
marked_urls = set()
marked_articles = set()

def scrape_pages(root, goal):
    if not root.startswith("https://vnexpress.net") or len(pages) == goal:
        return

    marked_urls.add(root)

    wait_time = 0
    response = requests.get(root)
    while response.status_code != 200:
        wait_time += 1
        if wait_time > 10:
            return
        time.sleep(wait_time)
        response = requests.get(root)

    page = BeautifulSoup(response.text, "lxml")

    page_type = page.find("meta", attrs={"name": 'tt_page_type'})
    article_id = page.find("meta", attrs={"name": 'tt_article_id'})
    if page_type != None and article_id != None:
        page_type_content = page_type["content"]
        article_id_content = article_id["content"]
        if page_type_content == "article" and article_id_content not in marked_articles:
            marked_articles.add(article_id_content)
            pages.append(page)
            print(root)
            print(page.title.string)

    if len(pages) == goal:
        return
    
    for sub_page in page.find_all("a"):
        if not sub_page.has_attr("href"):
            continue

        href = sub_page["href"]
        if not href.startswith("https://vnexpress.net") or "#" in href or href in marked_urls:
            continue

        scrape_pages(href, goal)
scrape_pages("https://vnexpress.net", 500)

https://vnexpress.net/iphone-18-pro-max-duoc-rao-chenh-9-trieu-dong-so-voi-gia-niem-yet-5121515.html
iPhone 18 Pro Max được rao chênh 9 triệu đồng so với giá niêm yết
https://vnexpress.net/bo-giao-duc-va-dao-tao-doi-y-lui-lich-siet-phuong-thuc-tuyen-sinh-dai-hoc-5121531.html
Đổi lộ trình siết phương thức tuyển sinh đại học 2027
https://vnexpress.net/du-kien-doi-cach-xet-tuyen-dai-hoc-2027-gioi-han-moi-nganh-chi-xet-bang-mot-phuong-thuc-5121397.html
Dự kiến siết phương thức xét tuyển đại học 2027
https://vnexpress.net/du-kien-doi-cach-xet-tuyen-dai-hoc-2027-gioi-han-moi-nganh-chi-xet-bang-mot-phuong-thuc-5121397-p2.html
Dự kiến 11 phương thức xét tuyển đại học từ năm 2027
https://vnexpress.net/mua-lu-lam-ngap-hon-2-200-nha-o-ngoai-thanh-ha-noi-5121540.html
Mưa lũ làm ngập hơn 2.200 nhà ở ngoại thành Hà Nội
https://vnexpress.net/sat-lo-vui-lap-ba-nguoi-trong-mot-gia-dinh-o-phu-tho-5121341.html
Sạt lở vùi lấp ba người trong một gia đình ở Phú Thọ
https://vnexpress.net/doc-luot-hieu-hoi-

KeyboardInterrupt: 

# Lưu raw data
Sau khi đã thu thập ít nhất 500 pages, lưu lại trong các folder đánh số gồm file html và metadata của trang

Lí do không extract luôn thông tin từ file HTML vì VnExpress không có cấu trúc định dạng consistent, mỗi trang tin có thể có layout khác nhau, vì thế nên sẽ lấy trước nguyên bản file HTML để khi clean data sẽ phân tích sau.

In [4]:
RAW_DATA_DIR = "../data/raw"
if not os.path.exists(RAW_DATA_DIR):
    os.makedirs(RAW_DATA_DIR)

for page in pages:
    metadata = {
        "url": page.find("meta", attrs={"name": "its_url"})["content"],
        "title": page.find("meta", attrs={"name": "its_title"})["content"],
        "sections": page.find("meta", attrs={"name": "its_subsection"})["content"],
        "tags": page.find("meta", attrs={"name": "its_tag"})["content"],
        "author": page.find("meta", attrs={"name": "its_author"})["content"],
        "word_count": int(page.find("meta", attrs={"name": "its_wordcount"})["content"]),
        "publication": int(page.find("meta", attrs={"name": "its_publication"})["content"]),
        "update_time": int(page.find("meta", attrs={"name": "article_updatetime"})["content"]),
        "description": page.find("meta", attrs={"name": "description"})["content"]
    }
    index = len(os.listdir(RAW_DATA_DIR))
    date = datetime.today().strftime('%Y-%m-%d %H:%M:%S')

    metadata_file = open(f'{RAW_DATA_DIR}/{index}_{date}.json', mode="w", encoding="utf-8")
    metadata_file.write(json.dumps(metadata, indent=4, ensure_ascii=False))
    metadata_file.close()